In [ ]:
spark.version

'4.1.0'

In [ ]:
## taking data from API and storing it in a pandas dataframe

import pandas as pd
import requests

url = 'https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol=IBM&apikey=*********'
r = requests.get(url)
data = r.json()



ts = data["Weekly Time Series"]

df = pd.DataFrame.from_dict(ts, orient="index")
df.index.name = "date"

df.reset_index(inplace=True)
df

,date,1. open,2. high,3. low,4. close,5. volume
0,2026-06-22,249.0600,253.3100,243.8100,252.5500,8916651
1,2026-06-18,272.0000,276.6200,243.6800,249.1000,34253212
2,2026-06-12,286.4400,290.5000,266.5000,272.2400,34602931
3,2026-06-05,322.5500,332.4600,281.0701,284.8400,86135011
4,2026-05-29,254.5500,300.9999,245.4500,297.8000,59445527
...,...,...,...,...,...,...
1385,1999-12-10,113.0000,122.1200,107.5600,109.0000,58626000
1386,1999-12-03,104.9400,112.8700,102.1200,111.8700,37670000
1387,1999-11-26,105.5000,109.8700,101.8100,105.0000,37165600
1388,1999-11-19,96.0000,105.1200,92.6200,103.9400,61550800


In [ ]:
access_key = "*************"
secret_key = "*************"
bucket = "stock1data1lake1garv"

In [ ]:
df_test = spark.read.csv(f"s3a://{bucket}/")
print("S3 Access works!")


S3 Access works!


In [ ]:
spark_df = spark.createDataFrame(df)


In [ ]:
spark_df

DataFrame[date: string, 1. open: string, 2. high: string, 3. low: string, 4. close: string, 5. volume: string]

In [ ]:
# writing as CSV
import datetime as dt


## this is for the file naming with date
fileappend = str(dt.datetime.now().date())
print(fileappend)


spark_df.write.mode("overwrite").option("header","true").option("fs.s3a.access.key", access_key).option("fs.s3a.secret.key", secret_key).option("fs.s3a.endpoint", "s3.amazonaws.com").csv(f"s3a://{bucket}/raw/ibm-{fileappend}")

print(f"Data written for {fileappend} to S3 successfully!")

################# Temp data writing....

2026-06-22
Data written for 2026-06-22 to S3 successfully!


In [ ]:
# Reading data directly from S3 WITH credentials
df = spark.read\
    .option("fs.s3a.access.key", access_key)\
    .option("fs.s3a.secret.key", secret_key)\
    .option("fs.s3a.endpoint", "s3.amazonaws.com")\
    .option("header", "true")\
    .csv(f"s3a://{bucket}/raw/ibm-{fileappend}")

display(df)

date,1. open,2. high,3. low,4. close,5. volume
2016-07-01,146.1800,152.9700,142.5000,152.3500,19155814
2016-06-24,152.6000,155.4800,146.1800,146.5900,22214411
2016-06-17,151.6300,152.7200,149.0000,151.9900,15849580
2016-06-10,153.0900,154.0900,151.8600,152.3700,14920842
2016-06-03,152.5600,153.8100,151.5400,152.8900,13470239
2016-05-27,147.6100,152.9300,146.6600,152.8400,14910875
2016-05-20,147.6500,149.9900,143.9550,147.2500,16406443
2016-05-13,147.7000,151.0900,147.0100,147.7200,17034303
2016-05-06,146.5600,147.9700,142.9000,147.2900,21059912
2016-04-29,148.1600,150.7800,144.1910,145.9400,16956550


In [ ]:
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- 1. open: string (nullable = true)
 |-- 2. high: string (nullable = true)
 |-- 3. low: string (nullable = true)
 |-- 4. close: string (nullable = true)
 |-- 5. volume: string (nullable = true)



In [ ]:
import pandas as pd 
import numpy as np 

df = df.toPandas()
df.head(6)

,date,1. open,2. high,3. low,4. close,5. volume
0,2016-07-01,146.1800,152.9700,142.5000,152.3500,19155814
1,2016-06-24,152.6000,155.4800,146.1800,146.5900,22214411
2,2016-06-17,151.6300,152.7200,149.0000,151.9900,15849580
3,2016-06-10,153.0900,154.0900,151.8600,152.3700,14920842
4,2016-06-03,152.5600,153.8100,151.5400,152.8900,13470239
5,2016-05-27,147.6100,152.9300,146.6600,152.8400,14910875


In [ ]:
df.describe()

,date,1. open,2. high,3. low,4. close,5. volume
count,1390,1390,1390,1390,1390,1390
unique,1390,1264,1310,1298,1291,1387
top,2016-07-01,113.0000,195.0000,101.0000,104.0000,18700700
freq,1,5,3,4,3,2


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1390 entries, 0 to 1389
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   date       1390 non-null   object
 1   1. open    1390 non-null   object
 2   2. high    1390 non-null   object
 3   3. low     1390 non-null   object
 4   4. close   1390 non-null   object
 5   5. volume  1390 non-null   object
dtypes: object(6)
memory usage: 65.3+ KB


In [ ]:
df.isnull().sum()

date         0
1. open      0
2. high      0
3. low       0
4. close     0
5. volume    0
dtype: int64

In [ ]:
df = df.dropna()
df = df.drop_duplicates()

In [ ]:
df

,date,1. open,2. high,3. low,4. close,5. volume
0,2016-07-01,146.1800,152.9700,142.5000,152.3500,19155814
1,2016-06-24,152.6000,155.4800,146.1800,146.5900,22214411
2,2016-06-17,151.6300,152.7200,149.0000,151.9900,15849580
3,2016-06-10,153.0900,154.0900,151.8600,152.3700,14920842
4,2016-06-03,152.5600,153.8100,151.5400,152.8900,13470239
...,...,...,...,...,...,...
1385,2003-04-11,82.6000,82.9000,78.1300,78.7500,39935800
1386,2003-04-04,79.2600,83.4800,78.1200,80.7900,46339300
1387,2003-03-28,82.4600,84.0000,80.5000,80.8500,42576300
1388,2003-03-21,78.0000,84.9000,77.8400,84.9000,58133300


In [ ]:
df['date'] = pd.to_datetime(df['date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1390 entries, 0 to 1389
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       1390 non-null   datetime64[ns]
 1   1. open    1390 non-null   object        
 2   2. high    1390 non-null   object        
 3   3. low     1390 non-null   object        
 4   4. close   1390 non-null   object        
 5   5. volume  1390 non-null   object        
dtypes: datetime64[ns](1), object(5)
memory usage: 65.3+ KB


In [ ]:
from pandas import to_numeric

df['1. open'] = pd.to_numeric(df['1. open'])
df['2. high'] = pd.to_numeric(df['2. high'])
df['3. low'] = pd.to_numeric(df['3. low'])
df['4. close'] = pd.to_numeric(df['4. close'])
df['5. volume'] = pd.to_numeric(df['5. volume'])

df = df.sort_values(by='date').reset_index(drop=True)

##Feature Engineering 

In [ ]:
# daily return by closing price with python fuc pct_change = percentage change from previous close
df["daily return"] = df['4. close'].pct_change()*100
df['daily return'] = df['daily return'].round(2)

# price range
df['price range'] = df['2. high'] - df['3. low']
df['price range'] = df['price range'].round(2)

# moving average by 5days 
df['ma5'] = df['4. close'].rolling(window=5).mean()
df['ma5'] = df['ma5'].round(2)

# moving average by 20days 
df['ma20'] = df['4. close'].rolling(window=20).mean()
df['ma20'] = df['ma20'].round(2)

print('\n ===New Feature Added===')
df['open'] = df['1. open']
df['high'] = df['2. high']
df['low'] = df['3. low']
df['close'] = df['4. close']
df['volume'] = df['5. volume']


 ===New Feature Added===


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1390 entries, 0 to 1389
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          1390 non-null   datetime64[ns]
 1   1. open       1390 non-null   float64       
 2   2. high       1390 non-null   float64       
 3   3. low        1390 non-null   float64       
 4   4. close      1390 non-null   float64       
 5   5. volume     1390 non-null   int64         
 6   daily return  1389 non-null   float64       
 7   price range   1390 non-null   float64       
 8   ma5           1386 non-null   float64       
 9   ma20          1371 non-null   float64       
 10  open          1390 non-null   float64       
 11  high          1390 non-null   float64       
 12  low           1390 non-null   float64       
 13  close         1390 non-null   float64       
 14  volume        1390 non-null   int64         
dtypes: datetime64[ns](1), float64(12), int

In [ ]:
df.drop(columns=['1. open','2. high','3. low','4. close','5. volume'],inplace=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1390 entries, 0 to 1389
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          1390 non-null   datetime64[ns]
 1   daily return  1389 non-null   float64       
 2   price range   1390 non-null   float64       
 3   ma5           1386 non-null   float64       
 4   ma20          1371 non-null   float64       
 5   open          1390 non-null   float64       
 6   high          1390 non-null   float64       
 7   low           1390 non-null   float64       
 8   close         1390 non-null   float64       
 9   volume        1390 non-null   int64         
dtypes: datetime64[ns](1), float64(8), int64(1)
memory usage: 108.7 KB


In [ ]:
df = df[['date','open','high','low','close','volume','daily return','price range','ma5','ma20']]

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1390 entries, 0 to 1389
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          1390 non-null   datetime64[ns]
 1   open          1390 non-null   float64       
 2   high          1390 non-null   float64       
 3   low           1390 non-null   float64       
 4   close         1390 non-null   float64       
 5   volume        1390 non-null   int64         
 6   daily return  1389 non-null   float64       
 7   price range   1390 non-null   float64       
 8   ma5           1386 non-null   float64       
 9   ma20          1371 non-null   float64       
dtypes: datetime64[ns](1), float64(8), int64(1)
memory usage: 108.7 KB


## Statistics & Insights

In [ ]:
print(f'Total days: {len(df)}')
print(f'Date range: {df['date'].min()} to {df['date'].max()}')
print(f'Average close price: ${df['close'].mean().round(2)}')
print(f'Highest close price: ${df['close'].max()}')
print(f'Lowest close price: ${df['close'].min()}')

Total days: 1390
Date range: 1999-11-12 00:00:00 to 2026-06-22 00:00:00
Average close price: $139.76
Highest close price: $309.24
Lowest close price: $56.6


In [ ]:
print(f'\n Average daily return: {df['daily return'].mean().round(2)}%')
print(f' Highest daily return: {df['daily return'].max().round(2)}%')
print(f' Lowest daily return: {df['daily return'].min().round(2)}%')
print(f'\n Volatility (STD): {df['daily return'].std().round(2)}%')


 Average daily return: 0.14%
 Highest daily return: 19.37%
 Lowest daily return: -15.49%

 Volatility (STD): 3.64%


## Finding Outliers

In [ ]:
threshold = 5

outliers = df[
    (df['daily return'] > threshold) | 
    (df['daily return'] < -threshold)
]

print(f'\n ===> Days with unusual returns (> {threshold}%')
print(f'\n Total number of outliears entries are {len(outliers)}')


 ===> Days with unusual returns (> 5%

 Total number of outliears entries are 180


### Logging number of processed rows

In [ ]:
from datetime import datetime
import json

run_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

try:
    rows_processed = int(df.shape[0])
    status = "SUCCESS"
    error_msg = ""
except Exception as e :
    rows_processed = 0
    status = "FAILED"
    error_msg = str(e)

log_entry = {
        "run_time" : run_time,
        "rows_processed" : rows_processed,
        "status" : status,
        "error_msg" : error_msg
}

log_path = f"s3a://{bucket}/logs/log_{run_time.replace(' ','_').replace(':','-')}.json"
spark.createDataFrame([log_entry]).coalesce(1).write.mode('overwrite') \
                .option('fs.s3a.access.key', access_key).option('fs.s3a.secret.key', secret_key).option('fs.s3a.endpoint','s3.amazonaws.com').json(log_path)

print(f'Log saved to {log_path}')
print(f'Rows {rows_processed} processed | Time: {run_time}')
               

/home/spark-d4a7e445-97fa-48aa-8790-ae/.ipykernel/9609/command-5679939643889279-814625950:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


Log saved to s3a://stock1data1lake1garv/logs/log_2026-06-22_20-39-27.json
Rows 1390 processed | Time: 2026-06-22 20:39:27


In [ ]:
df_final = spark.createDataFrame(df)

df_final.write.mode('overwrite')\
    .option('header','true').option('fs.s3a.access.key', access_key).option('fs.s3a.secret.key', secret_key).option('fs.s3a.endpoint','s3.amazonaws.com').csv(f's3a://{bucket}/processed/stock_cleaned-{fileappend}/',header=True)

print('clean data saved to S3!!!')

clean data saved to S3!!!


In [ ]:
df_final.createOrReplaceTempView('stock')


### Monthly Average Closing Price

In [ ]:
spark.sql('''
          select month(date) as month, 
          round(avg(close),2) as avg_close
          from stock
          group by month
          order by month
          ''').show()

+-----+---------+
|month|avg_close|
+-----+---------+
|    1|   142.21|
|    2|   141.99|
|    3|    140.9|
|    4|   139.64|
|    5|   142.64|
|    6|   141.26|
|    7|   137.95|
|    8|   138.72|
|    9|   137.89|
|   10|   138.67|
|   11|   137.33|
|   12|   137.95|
+-----+---------+



### Monthly Trading Volume 

In [ ]:
spark.sql("""
        select 
            month(date) as month,
            sum(volume) as total_volume
            from stock
            group by month
            order by month 
        """).show()

+-----+------------+
|month|total_volume|
+-----+------------+
|    1|  3707867831|
|    2|  3001278754|
|    3|  3686273855|
|    4|  3630509837|
|    5|  3214330708|
|    6|  3094719926|
|    7|  3258101067|
|    8|  2766345213|
|    9|  2791813831|
|   10|  3912203909|
|   11|  3118181928|
|   12|  3081195761|
+-----+------------+



### Highest Closing Price Per Month

In [ ]:
spark.sql("""
            select 
                month(date) as month,
                MAX(close)as highset_close
            from stock 
            group by month
            order by month 
            """).show()

+-----+-------------+
|month|highset_close|
+-----+-------------+
|    1|        306.7|
|    2|       298.93|
|    3|       261.54|
|    4|       253.47|
|    5|        297.8|
|    6|        289.7|
|    7|       291.97|
|    8|       250.05|
|    9|       284.31|
|   10|       307.46|
|   11|       308.58|
|   12|       309.24|
+-----+-------------+



### Most Volatile Month 

In [ ]:
spark.sql('''
          select 
                month(date) as month,
                round(avg(`price range`),2) as avg_volatility
                from stock 
                group by month
                order by avg_volatility desc ''').show()

+-----+--------------+
|month|avg_volatility|
+-----+--------------+
|   10|          8.26|
|    4|          7.73|
|    1|          7.56|
|    3|          6.76|
|    2|          6.67|
|    5|          6.63|
|    6|          6.36|
|    7|          6.22|
|   11|           6.1|
|    9|          5.57|
|   12|          5.38|
|    8|          5.37|
+-----+--------------+



### AVG Daily Return By Month 

In [ ]:
spark.sql("""
          select 
                month(date) as month,
                round(avg(`daily return`),2) as avg_return
                from stock 
                group by month
                order by month
          """).show()

+-----+----------+
|month|avg_return|
+-----+----------+
|    1|      0.75|
|    2|     -0.34|
|    3|       0.3|
|    4|     -0.12|
|    5|      0.36|
|    6|     -0.33|
|    7|      0.55|
|    8|     -0.19|
|    9|      0.03|
|   10|     -0.07|
|   11|      0.56|
|   12|      0.09|
+-----+----------+



### TOP 10 Trading Days 

In [ ]:
spark.sql("""
          select 
                date,
                `daily return`
                from stock 
                order by `daily return` desc
                limit 10 """).show()

+-------------------+------------+
|               date|daily return|
+-------------------+------------+
|2001-04-20 00:00:00|       19.37|
|2001-01-19 00:00:00|       18.59|
|2026-05-29 00:00:00|       17.32|
|2002-10-18 00:00:00|       16.16|
|2026-05-22 00:00:00|       15.75|
|2009-07-17 00:00:00|       14.47|
|2020-04-09 00:00:00|       14.26|
|2025-01-31 00:00:00|       13.75|
|2008-10-31 00:00:00|       13.28|
|2020-03-27 00:00:00|       13.25|
+-------------------+------------+



### Monthly Performance Dashboard Query 

In [ ]:
spark.sql("""
          select 
                month(date) as month,
                round(avg(close),2) as avg_close,
                sum(volume) as total_volume,
                round(avg(`daily return`),4) as avg_return,
                round(avg(`price range`),2) as avg_volatility
            from stock
            group by month
            order by month""").show()

+-----+---------+------------+----------+--------------+
|month|avg_close|total_volume|avg_return|avg_volatility|
+-----+---------+------------+----------+--------------+
|    1|   142.21|  3707867831|     0.754|          7.56|
|    2|   141.99|  3001278754|    -0.342|          6.67|
|    3|    140.9|  3686273855|    0.2967|          6.76|
|    4|   139.64|  3630509837|   -0.1222|          7.73|
|    5|   142.64|  3214330708|    0.3647|          6.63|
|    6|   141.26|  3094719926|   -0.3299|          6.36|
|    7|   137.95|  3258101067|    0.5531|          6.22|
|    8|   138.72|  2766345213|    -0.191|          5.37|
|    9|   137.89|  2791813831|    0.0322|          5.57|
|   10|   138.67|  3912203909|   -0.0684|          8.26|
|   11|   137.33|  3118181928|    0.5638|           6.1|
|   12|   137.95|  3081195761|    0.0866|          5.38|
+-----+---------+------------+----------+--------------+

